# Sesi 4 — Knowledge Agent dengan LLM Reasoning (Qwen, Structured Prompting)

Bagian dari kurikulum **Agentic Knowledge System — FastAPI + DuckDB + Qwen (Local, tanpa LLM cloud)**.

> Jalankan sel dari atas ke bawah secara berurutan. Sel bertanda `🔨 TODO` wajib dilengkapi peserta; solusi contoh disediakan pada sel setelahnya.

## 0. Setup Environment

In [ ]:
!pip -q install llama-cpp-python duckdb sentence-transformers fastapi httpx

## 1. Model Qwen Lokal
Model yang sama dipakai di seluruh kurikulum (Sesi 3, 6, 4, 7, 8) — cukup diunduh sekali per sesi Colab.

In [ ]:
import os

MODEL_PATH = "qwen2.5-0.5b-instruct-q4_k_m.gguf"
if not os.path.exists(MODEL_PATH):
    !wget -q -O {MODEL_PATH} "https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct-GGUF/resolve/main/qwen2.5-0.5b-instruct-q4_k_m.gguf"

from llama_cpp import Llama
llm = Llama(model_path=MODEL_PATH, n_ctx=4096, n_threads=4, verbose=False)

def qwen_llm_call(prompt, stop=None):
    out = llm(prompt, max_tokens=300, stop=stop)
    return out["choices"][0]["text"].strip()

## 2. Siapkan Ulang Knowledge Base & API (dari Sesi 1-2)

In [ ]:
import duckdb, uuid
from sentence_transformers import SentenceTransformer
from fastapi import FastAPI
from fastapi.testclient import TestClient

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
con = duckdb.connect("knowledge.duckdb")
con.execute("INSTALL vss; LOAD vss;")
con.execute("""
CREATE TABLE IF NOT EXISTS documents (
    id VARCHAR PRIMARY KEY, source VARCHAR, content TEXT,
    embedding FLOAT[384], created_at TIMESTAMP DEFAULT current_timestamp
);
""")
if con.execute("SELECT COUNT(*) FROM documents").fetchone()[0] == 0:
    seed = [("faq_produk.txt", "Garansi produk berlaku 1 tahun sejak tanggal pembelian."),
            ("sop_retur.txt", "Retur barang maksimal 7 hari setelah barang diterima.")]
    for source, text in seed:
        emb = embed_model.encode(text).tolist()
        con.execute("INSERT INTO documents VALUES (?, ?, ?, ?, current_timestamp)",
                    [str(uuid.uuid4()), source, text, emb])

app = FastAPI()

@app.get("/documents/search")
def search_documents(q: str, k: int = 5):
    q_emb = embed_model.encode(q).tolist()
    rows = con.execute("""
        SELECT id, source, content, array_distance(embedding, ?::FLOAT[384]) AS dist
        FROM documents ORDER BY dist ASC LIMIT ?
    """, [q_emb, k]).fetchall()
    return [{"id": r[0], "source": r[1], "content": r[2], "score": r[3]} for r in rows]

client = TestClient(app)

def call_tool(action, action_input):
    if action == "search_knowledge":
        return client.get("/documents/search", params={"q": action_input, "k": 3}).json()
    return {"error": "unknown tool"}

## 3. Kenapa Butuh Pola Baru?
Di **Sesi 3** kita pakai ReAct: loop iteratif `Thought/Action/Observation` yang bisa berkali-kali memanggil tool. Pola itu fleksibel tapi butuh beberapa kali panggilan LLM (lambat di CPU) dan rawan gagal parsing pada model kecil.

Di sesi ini kita coba pola **planner-executor**: satu prompt untuk *merencanakan* (apakah perlu tool, tool apa, query apa) dalam format JSON, lalu satu prompt terpisah untuk *menjawab* — ini mengemulasi 'function calling' pada model yang tidak punya kapabilitas tersebut secara native.

## 4. Prompt Planner (Emulasi Function Calling via JSON)

In [ ]:
import json, re

PLANNER_PROMPT = """Kamu adalah planner. Tugasmu memutuskan apakah pertanyaan berikut
butuh pencarian di knowledge base, dan menyusun keputusan dalam format JSON PERSIS seperti ini
(tanpa teks lain di luar JSON):

{{"need_tool": true, "tool": "search_knowledge", "query": "isi query pencarian"}}

atau jika tidak perlu tool:
{{"need_tool": false, "tool": null, "query": null}}

Pertanyaan: {question}
JSON:"""

def extract_json(text):
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return None

## 5. Fungsi Reasoning Agent (Single-shot Planner + Answer)

In [ ]:
def qwen_reasoning_agent(question, verbose=True, max_retries=2):
    plan = None
    for attempt in range(max_retries):
        plan_raw = qwen_llm_call(PLANNER_PROMPT.format(question=question), stop=["\n\n"])
        plan = extract_json(plan_raw)
        if verbose:
            print(f"[Planner attempt {attempt}] raw: {plan_raw!r} -> parsed: {plan}")
        if plan is not None:
            break

    context = ""
    if plan and plan.get("need_tool"):
        result = call_tool(plan["tool"], plan["query"])
        context = f"\nHasil pencarian: {result}\n"

    final_prompt = f"""Jawab pertanyaan berikut dalam Bahasa Indonesia yang singkat dan jelas,
gunakan konteks jika tersedia. Jangan mengulang instruksi ini dalam jawabanmu.
{context}
Pertanyaan: {question}
Jawaban:"""
    return qwen_llm_call(final_prompt, stop=["\n\n"])

print(qwen_reasoning_agent("Berapa lama masa garansi produk?"))

### 🔨 TODO 1
Tambahkan retry yang lebih baik: jika `extract_json` gagal terus setelah `max_retries`, jatuhkan ke jawaban fallback rule-based sederhana (misal: selalu panggil `search_knowledge` dengan seluruh teks pertanyaan sebagai query) alih-alih membiarkan `plan` bernilai `None`.

In [ ]:
# TODO: implementasikan fallback ketika planner gagal menghasilkan JSON valid
def qwen_reasoning_agent_v2(question, verbose=False):
    pass

**Solusi contoh:**
```python
def qwen_reasoning_agent_v2(question, verbose=False):
    plan_raw = qwen_llm_call(PLANNER_PROMPT.format(question=question), stop=["\n\n"])
    plan = extract_json(plan_raw)
    if plan is None:
        plan = {"need_tool": True, "tool": "search_knowledge", "query": question}
    context = ""
    if plan.get("need_tool"):
        result = call_tool(plan["tool"], plan["query"])
        context = f"\nHasil pencarian: {result}\n"
    final_prompt = f"Jawab dalam Bahasa Indonesia.\n{context}\nPertanyaan: {question}\nJawaban:"
    return qwen_llm_call(final_prompt, stop=["\n\n"])
```

### 🔨 TODO 2 — Perbandingan ReAct (Sesi 3) vs Planner (Sesi ini)
Jalankan 5 pertanyaan yang sama lewat `react_loop` (tempel ulang dari Sesi 3) dan `qwen_reasoning_agent`. Catat: jumlah panggilan LLM, waktu total, dan apakah jawaban sama-sama benar.

In [ ]:
import pandas as pd, time

questions = [
    "Berapa lama masa garansi produk?",
    "Apa syarat retur barang?",
]

results = []
for q in questions:
    t0 = time.time()
    ans = qwen_reasoning_agent(q, verbose=False)
    results.append({"question": q, "pattern": "planner", "answer": ans, "latency_s": round(time.time() - t0, 2)})

pd.DataFrame(results)

## Ringkasan Sesi 4
Knowledge Agent kini punya dua pola prompting di atas model Qwen yang sama: **ReAct iteratif** (Sesi 3, cocok untuk tugas multi-langkah) dan **planner-executor JSON** (sesi ini, lebih cepat untuk tugas satu langkah). Sesi 5 memulai domain baru: **Knowledge ERP**.